# Iterative Fisher LDA deflation of activation source folders

This notebook streams activation batches except for the `abst` source folder, excludes rows whose `base_unit` is `millennia` or `seconds`, and mean-aggregates activations by `(time_horizon_months, source_folder, task)`. The aggregates are projected onto their first 128 principal components in float64, then mutually orthogonal source-folder Fisher LDA directions are removed in PC space until `MAX_DEFLATIONS` is reached or no numerically independent direction remains. In addition to per-direction variance, monotonic between-centroid geometry is tracked, and a self-contained inference bundle is saved for every fitted state under `results/deflation_artifacts/iter_*`.

In [1]:
from collections import defaultdict
from pathlib import Path
import shutil
import sys

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

ROOT = Path.cwd().resolve()
if not (ROOT / '.acts').is_dir():
    ROOT = ROOT.parent
if not (ROOT / '.acts').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks/.')

src_path = str(ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from temporal_manifolds.horizon.cache import time_horizon_months

ACTS_DIR = ROOT / '.acts'
ARTIFACTS_DIR = ROOT / 'results' / 'deflation_artifacts'
EXCLUDED_SOURCE_FOLDERS = {'abst'}
EXCLUDED_BASE_UNITS = {} # 'millennia', 'seconds'
POSITION_INDEX = 0  # cached files contain the selected final-token position here
N_PCA_COMPONENTS = 128
MAX_DEFLATIONS = 120  # User-set upper bound; fitting may stop earlier.
DIRECTION_RESIDUAL_RELATIVE_TOL = 1e-8
MONOTONICITY_RTOL = 1e-10
MONOTONICITY_ATOL = 1e-12
DEFLATION_CHUNK_ROWS = 128
ALPHA = 1.0
if not 0 <= MAX_DEFLATIONS < N_PCA_COMPONENTS:
    raise ValueError(
        f'MAX_DEFLATIONS must be between 0 and {N_PCA_COMPONENTS - 1}; '
        f'found {MAX_DEFLATIONS}.'
    )
if ARTIFACTS_DIR.exists():
    shutil.rmtree(ARTIFACTS_DIR)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cleared previous artifacts: {ARTIFACTS_DIR}')
print(f'Input: {ACTS_DIR}')

Cleared previous artifacts: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts
Input: C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\.acts


## Stream, filter, and mean-aggregate

Each batch is released as soon as it has been accumulated. The only activation tensors retained across batches are one running sum per `(time_horizon_months, source_folder, task)` group.

In [2]:
batch_paths = sorted(
    path
    for path in ACTS_DIR.rglob('activations_batch_*.pt')
    if path.parent.relative_to(ACTS_DIR).as_posix() not in EXCLUDED_SOURCE_FOLDERS
)
if not batch_paths:
    raise FileNotFoundError(f'No activation batches found beneath {ACTS_DIR}')

group_sums = {}
group_counts = defaultdict(int)
component_name = None
feature_count = None
rows_seen = rows_excluded = rows_invalid = rows_kept = 0

for file_number, path in enumerate(batch_paths, start=1):
    payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
    activations = payload['activations']
    names = list(activations)
    if len(names) != 1:
        raise ValueError(f'Expected one activation component in {path}, found {names}')
    if component_name is None:
        component_name = names[0]
    elif names[0] != component_name:
        raise ValueError(f'Inconsistent component in {path}: {names[0]} != {component_name}')

    tensor = activations[component_name]
    metadata_rows = payload['prompt_metadata']
    if tensor.ndim != 3 or tensor.shape[0] != len(metadata_rows):
        raise ValueError(f'Misaligned activations and metadata in {path}')
    values = tensor[:, POSITION_INDEX, :].to(dtype=torch.float32)
    if feature_count is None:
        feature_count = int(values.shape[1])
    elif values.shape[1] != feature_count:
        raise ValueError(f'Inconsistent activation width in {path}')

    source_folder = path.parent.relative_to(ACTS_DIR).as_posix()
    rows_seen += len(metadata_rows)
    for row_index, metadata in enumerate(metadata_rows):
        base_unit = str(metadata.get('base_unit', '')).strip().casefold()
        if base_unit in EXCLUDED_BASE_UNITS:
            rows_excluded += 1
            continue

        task_value = metadata.get('task')
        task = '' if task_value is None else str(task_value).strip()
        horizon = time_horizon_months(metadata.get('base_value'), metadata.get('base_unit'))
        if not task or not np.isfinite(horizon):
            rows_invalid += 1
            continue

        key = (float(horizon), source_folder, task)
        if key in group_sums:
            group_sums[key].add_(values[row_index])
        else:
            group_sums[key] = values[row_index].clone()
        group_counts[key] += 1
        rows_kept += 1

    # Drop every reference to the memory-mapped batch before opening the next file.
    del payload, activations, tensor, values, metadata_rows
    if file_number % 100 == 0 or file_number == len(batch_paths):
        print(f'Processed {file_number:,}/{len(batch_paths):,} batches', end='\r')

del batch_paths
if not group_sums:
    raise ValueError('No activation rows remain after filtering.')
print()
print(
    f'Rows seen: {rows_seen:,}; excluded by unit: {rows_excluded:,}; '
    f'invalid: {rows_invalid:,}; aggregated: {rows_kept:,}'
)
print(f'Groups: {len(group_sums):,}; activation width: {feature_count:,}; component: {component_name}')

Processed 1,867/1,867 batches
Rows seen: 238,393; excluded by unit: 0; invalid: 1,004; aggregated: 237,389
Groups: 6,425; activation width: 2,560; component: layer_out/21


In [3]:
keys = sorted(group_sums, key=lambda key: (key[0], key[1], key[2]))
X_current = np.empty((len(keys), feature_count), dtype=np.float64)
source_sample_counts = np.empty(len(keys), dtype=np.int64)

for group_index, key in enumerate(keys):
    count = group_counts.pop(key)
    sum_vector = group_sums.pop(key)
    sum_vector.div_(count)
    X_current[group_index] = sum_vector.numpy()
    source_sample_counts[group_index] = count
    del sum_vector

groups = pd.DataFrame(
    keys,
    columns=['time_horizon_months', 'source_folder', 'task'],
)
groups['source_sample_count'] = source_sample_counts
del keys, source_sample_counts, group_sums, group_counts

display(
    groups.groupby('source_folder').agg(
        horizon_task_groups=('source_folder', 'size'),
        tasks=('task', 'nunique'),
        raw_rows=('source_sample_count', 'sum'),
    )
)
display(groups.head())
print(f'Aggregated activation matrix: {X_current.shape}')
assert X_current.ndim == 2 and np.isfinite(X_current).all()

,horizon_task_groups,tasks,raw_rows
source_folder,,,
indexed,1285,50,25146
indirect,1285,50,32921
new_conv,1285,50,48906
plain,1285,50,65208
plain_long,1285,50,65208


,time_horizon_months,source_folder,task,source_sample_count
0,3.802570e-07,indexed,activate a building's emergency alarm after co...,18
1,3.802570e-07,indexed,answer a yes-or-no question,18
2,3.802570e-07,indexed,choose between two lunch options,18
3,3.802570e-07,indexed,copy a short code from one screen to another,18
4,3.802570e-07,indexed,press a button when a light turns green,18


Aggregated activation matrix: (6425, 2560)


## Project to 128 principal components

PCA is fit on the mean-aggregated activation matrix without whitening. After the PC scores are computed, the original activation-space matrix is discarded, while the fitted PCA model is retained for the per-iteration inference bundles. All subsequent LDA scores and deflations remain in this 128-dimensional PC space; no inverse transform is performed.

In [4]:
if min(X_current.shape) < N_PCA_COMPONENTS:
    raise ValueError(
        f'PCA requires at least {N_PCA_COMPONENTS} rows and features; found {X_current.shape}.'
    )

pca = PCA(
    n_components=N_PCA_COMPONENTS,
    whiten=False,
    svd_solver='randomized',
    random_state=0,
    copy=False,
)
X_pc = pca.fit_transform(X_current)
retained_variance = float(pca.explained_variance_ratio_.sum())
del X_current
X_current = np.asarray(X_pc, dtype=np.float64)
del X_pc

print(f'PC-space matrix: {X_current.shape}')
print(f'Variance retained by 128 PCs: {retained_variance:.2%}')
assert X_current.shape[1] == N_PCA_COMPONENTS and np.isfinite(X_current).all()

PC-space matrix: (6425, 128)
Variance retained by 128 PCs: 99.32%


## Orthogonal leading-component LDA deflations

At each iteration, `source_folder` is the class label and Fisher LDA is fit with exactly one component in the 128-dimensional PC space. The candidate direction is re-orthogonalized twice against every previously removed direction before it is normalized. If the orthogonal residual is too small, fitting stops; otherwise the centered score is $s_i = (x - \bar{x})^T \hat{w}_i$ and the next PC-space input is computed in place as $x_{new} = x - s_i \hat{w}_i$. Keeping the matrix in float64 and maintaining an orthonormal removed-direction basis prevents numerical cycling.

Within-class variance is the equally weighted mean of the sample variance of scores in each source folder. Across-class variance is the sample variance of source-folder mean scores along the newly selected direction, so it need not be monotonic because that direction changes. Total between-class scatter and mean pairwise centroid distance are also recorded; both must be non-increasing under each orthogonal projection. During notebook initialization, the entire previous artifact directory is removed before data loading begins. Every saved bundle keeps the existing PCA plus `completed_deflations` inference contract and adds stopping and geometry metadata.

In [5]:
source_class, source_values = pd.factorize(groups['source_folder'], sort=True)
class_counts = np.bincount(source_class, minlength=len(source_values))
if len(source_values) < 2:
    raise ValueError('Fisher LDA requires at least two source folders.')
if (class_counts < 2).any():
    sparse = np.asarray(source_values)[class_counts < 2]
    raise ValueError(f'Every source folder needs at least two aggregates; singleton folders: {sparse}')

class_row_indices = [
    np.flatnonzero(source_class == class_id)
    for class_id in range(len(source_values))
]

def class_geometry(values):
    class_centroids = np.vstack([
        values[row_indices].mean(axis=0, dtype=np.float64)
        for row_indices in class_row_indices
    ])
    centered_centroids = class_centroids - class_centroids.mean(axis=0)
    total_between_class_scatter = float(
        np.square(centered_centroids).sum(dtype=np.float64) / (len(class_centroids) - 1)
    )
    pairwise_differences = class_centroids[:, None, :] - class_centroids[None, :, :]
    upper_triangle = np.triu_indices(len(class_centroids), k=1)
    mean_pairwise_centroid_distance = float(
        np.linalg.norm(pairwise_differences, axis=2)[upper_triangle].mean()
    )
    return total_between_class_scatter, mean_pairwise_centroid_distance

variance_records = []
deflation_steps = []
removed_directions = []
previous_between_scatter = None
previous_centroid_distance = None
initial_between_scatter = None
for deflations_completed in range(MAX_DEFLATIONS + 1):
    total_between_scatter, mean_centroid_distance = class_geometry(X_current)
    if initial_between_scatter is None:
        initial_between_scatter = total_between_scatter
    if previous_between_scatter is not None:
        scatter_tolerance = (
            MONOTONICITY_ATOL + MONOTONICITY_RTOL * abs(previous_between_scatter)
        )
        distance_tolerance = (
            MONOTONICITY_ATOL + MONOTONICITY_RTOL * abs(previous_centroid_distance)
        )
        if total_between_scatter > previous_between_scatter + scatter_tolerance:
            raise RuntimeError(
                'Total between-class scatter increased after an orthogonal deflation: '
                f'{previous_between_scatter:.12g} -> {total_between_scatter:.12g}.'
            )
        if mean_centroid_distance > previous_centroid_distance + distance_tolerance:
            raise RuntimeError(
                'Mean pairwise centroid distance increased after an orthogonal deflation: '
                f'{previous_centroid_distance:.12g} -> {mean_centroid_distance:.12g}.'
            )
    previous_between_scatter = total_between_scatter
    previous_centroid_distance = mean_centroid_distance

    lda = LinearDiscriminantAnalysis(n_components=1, solver='svd')
    lda.fit(X_current, source_class)

    raw_direction = np.asarray(lda.scalings_[:, 0], dtype=np.float64).copy()
    raw_direction_norm = float(np.linalg.norm(raw_direction))
    if not np.isfinite(raw_direction_norm) or raw_direction_norm == 0.0:
        raise ValueError(
            f'State after {deflations_completed} deflations produced an invalid first LDA component.'
        )
    candidate_direction = raw_direction / raw_direction_norm
    del raw_direction

    if removed_directions:
        removed_basis = np.column_stack(removed_directions)
        # Two modified Gram-Schmidt passes keep the new direction orthogonal in float64.
        for _ in range(2):
            candidate_direction -= removed_basis @ (removed_basis.T @ candidate_direction)
    else:
        removed_basis = np.empty((N_PCA_COMPONENTS, 0), dtype=np.float64)

    direction_residual_fraction = float(np.linalg.norm(candidate_direction))
    direction_available = (
        np.isfinite(direction_residual_fraction)
        and direction_residual_fraction > DIRECTION_RESIDUAL_RELATIVE_TOL
    )
    lda_center = np.asarray(lda.xbar_, dtype=np.float64).copy()

    if direction_available:
        w1_unit = candidate_direction / direction_residual_fraction
        orthogonality_error = (
            float(np.max(np.abs(removed_basis.T @ w1_unit)))
            if removed_directions
            else 0.0
        )
        if orthogonality_error > 1e-10:
            raise RuntimeError(
                f'Orthogonalization error {orthogonality_error:.3e} exceeds tolerance.'
            )

        # Equivalent to (X_current - lda.xbar_) @ w1_unit without a centered matrix.
        s1 = X_current @ w1_unit
        s1 -= np.dot(lda_center, w1_unit)
        within_variance_by_class = np.empty(len(source_values), dtype=np.float64)
        class_mean_scores = np.empty(len(source_values), dtype=np.float64)
        for class_id, row_indices in enumerate(class_row_indices):
            class_scores = s1[row_indices]
            within_variance_by_class[class_id] = class_scores.var(ddof=1, dtype=np.float64)
            class_mean_scores[class_id] = class_scores.mean(dtype=np.float64)
        mean_within_variance = float(within_variance_by_class.mean())
        across_class_variance = float(class_mean_scores.var(ddof=1))
    else:
        w1_unit = None
        s1 = None
        orthogonality_error = np.nan
        mean_within_variance = np.nan
        across_class_variance = np.nan

    if not direction_available:
        stopping_reason = 'orthogonal_direction_residual_below_tolerance'
    elif deflations_completed == MAX_DEFLATIONS:
        stopping_reason = 'maximum_deflations_reached'
    else:
        stopping_reason = None

    record = {
        'deflations_completed': deflations_completed,
        'stage': (
            'Pre-deflation'
            if deflations_completed == 0
            else f'After deflation {deflations_completed}'
        ),
        'mean_within_class_variance': mean_within_variance,
        'across_class_variance': across_class_variance,
        'total_between_class_scatter': total_between_scatter,
        'between_scatter_fraction_remaining': (
            total_between_scatter / initial_between_scatter
            if initial_between_scatter > 0.0
            else 0.0
        ),
        'mean_pairwise_centroid_distance': mean_centroid_distance,
        'direction_residual_fraction': direction_residual_fraction,
        'orthogonality_error': orthogonality_error,
        'stopping_reason': stopping_reason,
    }
    variance_records.append(record)

    # Keep completed_deflations unchanged for downstream inference compatibility.
    iteration_dir = ARTIFACTS_DIR / f'iter_{deflations_completed}'
    iteration_dir.mkdir(parents=True, exist_ok=True)
    artifact = {
        'artifact_version': 2,
        'deflations_completed': deflations_completed,
        'maximum_deflations': MAX_DEFLATIONS,
        'direction_residual_relative_tolerance': DIRECTION_RESIDUAL_RELATIVE_TOL,
        'stopping_reason': stopping_reason,
        'next_deflation_available': bool(direction_available and stopping_reason is None),
        'activation_component': component_name,
        'position_index': POSITION_INDEX,
        'input_feature_count': feature_count,
        'deflation_alpha': ALPHA,
        'aggregation_fields': ('time_horizon_months', 'source_folder', 'task'),
        'excluded_source_folders': tuple(sorted(EXCLUDED_SOURCE_FOLDERS)),
        'excluded_base_units': tuple(sorted(EXCLUDED_BASE_UNITS)),
        'source_labels': np.asarray(source_values).copy(),
        'pca': pca,
        'completed_deflations': [
            {
                'center': step['center'].copy(),
                'direction': step['direction'].copy(),
                'alpha': step['alpha'],
            }
            for step in deflation_steps
        ],
        'orthonormal_deflation_basis': removed_basis.copy(),
        'lda': lda,
        'lda_center': lda_center.copy(),
        'lda_score_direction': None if w1_unit is None else w1_unit.copy(),
        'diagnostics': record.copy(),
    }
    artifact_path = iteration_dir / 'lda_pipeline.joblib'
    joblib.dump(artifact, artifact_path, compress=3)
    del artifact
    print(f'Saved iteration {deflations_completed} artifacts to {artifact_path}')

    if stopping_reason is not None:
        print(f'Stopped after {deflations_completed} deflations: {stopping_reason}.')
        break

    for start in range(0, X_current.shape[0], DEFLATION_CHUNK_ROWS):
        stop = min(start + DEFLATION_CHUNK_ROWS, X_current.shape[0])
        update = s1[start:stop, None] * w1_unit[None, :]
        X_current[start:stop] -= ALPHA * update

    deflation_steps.append({
        'center': lda_center.copy(),
        'direction': w1_unit.copy(),
        'alpha': ALPHA,
    })
    removed_directions.append(w1_unit.copy())
    print(
        f'Completed LDA deflation {deflations_completed + 1}/{MAX_DEFLATIONS}',
        end='\r',
    )

variance_history = pd.DataFrame.from_records(variance_records)
del variance_records, X_current, class_row_indices, source_class, source_values, class_counts, groups
del pca, deflation_steps, removed_directions
print()
display(variance_history)

Saved iteration 0 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_0\lda_pipeline.joblib
Saved iteration 1 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_1\lda_pipeline.joblib
Saved iteration 2 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_2\lda_pipeline.joblib
Saved iteration 3 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_3\lda_pipeline.joblib
Saved iteration 4 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_4\lda_pipeline.joblib
Saved iteration 5 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifacts\iter_5\lda_pipeline.joblib
Saved iteration 6 artifacts to C:\Users\91967\Desktop\AISC\temporal-manifolds-last-position\results\deflation_artifact

,deflations_completed,stage,mean_within_class_variance,across_class_variance,total_between_class_scatter,between_scatter_fraction_remaining,mean_pairwise_centroid_distance,direction_residual_fraction,orthogonality_error,stopping_reason
0,0,Pre-deflation,0.006289,23.467516,169.974429,1.000000,17.884063,1.000000e+00,0.000000e+00,NaN
1,1,After deflation 1,0.010373,9.277318,146.506913,0.861935,16.669301,9.949180e-01,5.030698e-17,NaN
2,2,After deflation 2,0.007726,4.459304,137.229596,0.807354,16.093863,9.901659e-01,2.298509e-17,NaN
3,3,After deflation 3,0.009981,3.321574,132.770292,0.781119,15.780008,9.757341e-01,5.464379e-17,NaN
4,4,After deflation 4,0.084276,12.403963,129.448718,0.761578,15.562501,8.124891e-01,1.387779e-16,NaN
...,...,...,...,...,...,...,...,...,...,...
105,105,After deflation 105,1.695899,0.023395,0.334879,0.001970,0.725278,9.870981e-08,7.714457e-17,NaN
106,106,After deflation 106,1.289975,0.016853,0.311484,0.001833,0.698806,4.278705e-08,1.106319e-16,NaN
107,107,After deflation 107,3.547510,0.042037,0.294631,0.001733,0.670286,1.490190e-08,1.020843e-16,NaN
108,108,After deflation 108,4.587364,0.050114,0.252594,0.001486,0.626700,2.211741e-08,5.907007e-17,NaN


In [6]:
fig = make_subplots(
    rows=2,
    cols=1,
    specs=[[{'secondary_y': True}], [{'secondary_y': True}]],
    subplot_titles=(
        'Variance along the newly selected orthogonal LDA direction',
        'Monotonic source-centroid geometry',
    ),
)
fig.add_trace(
    go.Scatter(
        x=variance_history['deflations_completed'],
        y=variance_history['mean_within_class_variance'],
        name='Mean within-class variance',
        mode='lines+markers',
    ),
    row=1,
    col=1,
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=variance_history['deflations_completed'],
        y=variance_history['across_class_variance'],
        name='Across-class variance',
        mode='lines+markers',
    ),
    row=1,
    col=1,
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=variance_history['deflations_completed'],
        y=variance_history['total_between_class_scatter'],
        name='Total between-class scatter',
        mode='lines+markers',
    ),
    row=2,
    col=1,
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=variance_history['deflations_completed'],
        y=variance_history['mean_pairwise_centroid_distance'],
        name='Mean pairwise centroid distance',
        mode='lines+markers',
    ),
    row=2,
    col=1,
    secondary_y=True,
)
fig.update_xaxes(
    title_text='Deflations completed (0 = pre-deflation)',
    dtick=1,
    row=2,
    col=1,
)
fig.update_yaxes(
    title_text='Mean within-class variance',
    rangemode='tozero',
    row=1,
    col=1,
    secondary_y=False,
)
fig.update_yaxes(
    title_text='Across-class variance',
    rangemode='tozero',
    row=1,
    col=1,
    secondary_y=True,
)
fig.update_yaxes(
    title_text='Total between-class scatter',
    rangemode='tozero',
    row=2,
    col=1,
    secondary_y=False,
)
fig.update_yaxes(
    title_text='Mean centroid distance',
    rangemode='tozero',
    row=2,
    col=1,
    secondary_y=True,
)
fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    legend_title_text='',
    height=850,
    title='Orthogonal Fisher LDA deflation diagnostics',
    xaxis=dict(autorange=True),
    yaxis=dict(autorange=True)
)
fig.show()